# 🎓 Ablation Study: OpenFace Geometric Stream Alone (Binary Engagement)

### 🔬 Objective
In our primary architecture, we used a **Dual-Stream** model (EfficientNet-B0 appearance features + OpenFace geometric features) passed through a **Transformer Encoder**.

In this **Ablation Study**, we systematically analyze the contribution of facial geometric descriptors by:
1. **Removing EfficientNet-B0** (no deep appearance stream)
2. **Removing the Transformer Encoder** (no complex self-attention)
3. **Using OpenFace Alone** (head pose, eye gaze, action units, landmarks)
4. Retaining the **same classification head** (Temporal Mean Pooling $\to$ FC $\to$ ReLU $\to$ Dropout $\to$ FC $\to$ Softmax)
5. Evaluating on **2 classes**: `Engaged` vs `Disengaged`

---

## 📌 Step 1: Install Dependencies & Verify GPU / CPU Runtime

In [ ]:
# Install required libraries
!pip install -q torch torchvision scikit-learn matplotlib pandas numpy

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ Hardware accelerator: {device}")
if torch.cuda.is_available():
    print(f"   GPU Device Name: {torch.cuda.get_device_name(0)}")

## 📌 Step 2: Set Up Dataset Path (`sequence_fused_features_binary`)

Choose **Option A** (Google Drive mount - recommended) or **Option B** (Direct Colab upload).

In [ ]:
import os
import shutil

# ====================================================================
# OPTION A: If you uploaded the folder/zip to your Google Drive
# ====================================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
except Exception as e:
    print("Not running inside Colab or drive mounting skipped.")

# Potential paths where sequence_fused_features_binary might be located
candidate_dirs = [
    "/content/sequence_fused_features_binary",
    "/content/drive/MyDrive/sequence_fused_features_binary",
    "/content/drive/MyDrive/Student_Engagement_DL/sequence_fused_features_binary",
    "/content/dataset/sequence_fused_features_binary"
]

DATASET_DIR = None
for c in candidate_dirs:
    if os.path.exists(c) and len([f for f in os.listdir(c) if f.endswith('.pt') or f.endswith('.npy')]) > 0:
        DATASET_DIR = c
        break

# If you uploaded a zip file named 'sequence_fused_features_binary.zip', auto-unzip it:
zip_candidates = [
    "/content/sequence_fused_features_binary.zip",
    "/content/drive/MyDrive/sequence_fused_features_binary.zip",
    "/content/drive/MyDrive/Student_Engagement_DL/sequence_fused_features_binary.zip"
]
for z in zip_candidates:
    if os.path.exists(z) and DATASET_DIR is None:
        print(f"Unzipping {z} to /content/sequence_fused_features_binary...")
        !unzip -q {z} -d /content/
        DATASET_DIR = "/content/sequence_fused_features_binary"
        break

if DATASET_DIR is None:
    DATASET_DIR = "/content/sequence_fused_features_binary"
    print(f"⚠️ Folder not automatically located yet. Set DATASET_DIR to your folder path.")
else:
    n_files = len([f for f in os.listdir(DATASET_DIR) if f.endswith('.pt') or f.endswith('.npy')])
    print(f"✅ Dataset folder found at: {DATASET_DIR} ({n_files} feature files detected)")

## 📌 Step 3: Inspect Features & Extract OpenFace Geometric Stream
This cell inspects sample files to verify tensor dimensions, slices **strictly OpenFace features** (dropping EfficientNet), and checks the 2-class label distribution (`Engaged` vs `Disengaged`).

In [ ]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class OpenFaceBinaryDataset(Dataset):
    def __init__(self, file_paths, max_seq_len=120):
        self.samples = []
        self.max_seq_len = max_seq_len
        self._load_samples(file_paths)

    def _extract_sample(self, path):
        if path.endswith(".pt"):
            data = torch.load(path, map_location="cpu")
        elif path.endswith(".npy"):
            data = np.load(path, allow_pickle=True)
            if data.dtype == object and data.size == 1:
                data = data.item()
        else:
            return None, None

        # 1. Feature tensor extraction
        raw_feat = None
        if isinstance(data, dict):
            for k in ["sequence", "feature", "features", "fused_seq", "data"]:
                if k in data:
                    raw_feat = data[k]
                    break
        elif isinstance(data, (torch.Tensor, np.ndarray)):
            raw_feat = data

        if raw_feat is None:
            return None, None

        if isinstance(raw_feat, np.ndarray):
            raw_feat = torch.from_numpy(raw_feat).float()
        else:
            raw_feat = raw_feat.float()

        if raw_feat.dim() == 1:
            raw_feat = raw_feat.unsqueeze(0) # [1, Dim]

        # 2. Extract strictly OpenFace features (Ablation: drop EfficientNet)
        n_frames, feat_dim = raw_feat.shape
        if feat_dim > 500:
            # Fused feature (1408 = 1280 CNN + 128 OpenFace or 128 OpenFace + 1280 CNN)
            # Slice 128-dim OpenFace portion
            if feat_dim == 1408 and raw_feat.shape[1] > 1280:
                openface_feat = raw_feat[:, -128:]
            else:
                openface_feat = raw_feat[:, :128]
        else:
            openface_feat = raw_feat

        # 3. Label normalization to 2 classes: 0 (Engaged), 1 (Disengaged)
        raw_label = None
        if isinstance(data, dict) and "label" in data:
            raw_label = data["label"]
        else:
            fn = os.path.basename(path).lower()
            raw_label = 1 if ("disengaged" in fn or "disengage" in fn) else 0

        if isinstance(raw_label, str):
            lbl_str = raw_label.lower().strip()
            bin_label = 1 if ("disengaged" in lbl_str or "disengage" in lbl_str) else 0
        else:
            bin_label = 0 if int(raw_label) == 0 else 1

        return openface_feat, bin_label

    def _load_samples(self, file_paths):
        for p in file_paths:
            try:
                feat, lbl = self._extract_sample(p)
                if feat is not None:
                    self.samples.append({"path": p, "feature": feat, "label": lbl})
            except Exception:
                continue

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        feat = s["feature"]
        if feat.shape[0] > self.max_seq_len:
            feat = feat[:self.max_seq_len]
        return feat, s["label"]

def collate_seqs(batch):
    seqs, labels = zip(*batch)
    batch_size = len(seqs)
    lens = [s.shape[0] for s in seqs]
    max_len = max(lens)
    dim = seqs[0].shape[1]
    padded = torch.zeros((batch_size, max_len, dim), dtype=torch.float32)
    for i, s in enumerate(seqs):
        padded[i, :s.shape[0], :] = s
    return padded, torch.tensor(lens, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

# Verify and inspect sample files
sample_files = [os.path.join(DATASET_DIR, f) for f in os.listdir(DATASET_DIR) if f.endswith('.pt') or f.endswith('.npy')]
print(f"Found {len(sample_files)} total files.")

test_dataset = OpenFaceBinaryDataset(sample_files)
labels = [s["label"] for s in test_dataset.samples]
feature_dim = test_dataset.samples[0]["feature"].shape[1]

print(f"✅ Successfully processed {len(test_dataset)} samples.")
print(f"   • Engaged (0): {labels.count(0)}")
print(f"   • Disengaged (1): {labels.count(1)}")
print(f"   • Extracted OpenFace Feature Dimension: {feature_dim}")

## 📌 Step 4: Define the Ablation Architecture
The model consists of:
1. **Input**: OpenFace features alone
2. **Temporal Representation**: Mean Pooling over frames (Step 5 in your diagram)
3. **Classification Head**: `Linear(input_dim -> 128)` $\to$ `ReLU()` $\to$ `Dropout(0.3)` $\to$ `Linear(128 -> 2)`

In [ ]:
import torch.nn as nn

class OpenFaceAblationClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.3, num_classes=2):
        super(OpenFaceAblationClassifier, self).__init__()
        # Classification Head directly from the architecture
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x, seq_lens=None):
        if x.dim() == 2:
            pooled = x
        elif x.dim() == 3:
            if seq_lens is not None:
                batch_size, max_len, _ = x.shape
                mask = torch.arange(max_len, device=x.device).unsqueeze(0) < seq_lens.unsqueeze(1)
                mask = mask.unsqueeze(-1).float()
                sum_x = (x * mask).sum(dim=1)
                lens = seq_lens.clamp(min=1).unsqueeze(1).float()
                pooled = sum_x / lens
            else:
                pooled = x.mean(dim=1)
        return self.classifier(pooled)

# Create model instance
model = OpenFaceAblationClassifier(input_dim=feature_dim, hidden_dim=128, dropout=0.3, num_classes=2).to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal Trainable Parameters: {n_params:,}")

## 📌 Step 5: Split Data & Train Model (Track Best Validation Accuracy)

In [ ]:
import torch.optim as optim

# Stratified Split: 70% Train, 15% Validation, 15% Test
valid_paths = [s["path"] for s in test_dataset.samples]
valid_labels = [s["label"] for s in test_dataset.samples]

try:
    train_p, temp_p, train_l, temp_l = train_test_split(valid_paths, valid_labels, test_size=0.30, stratify=valid_labels, random_state=42)
    val_p, test_p, val_l, test_l = train_test_split(temp_p, temp_l, test_size=0.50, stratify=temp_l, random_state=42)
except Exception:
    train_p, temp_p = train_test_split(valid_paths, test_size=0.30, random_state=42)
    val_p, test_p = train_test_split(temp_p, test_size=0.50, random_state=42)

train_loader = DataLoader(OpenFaceBinaryDataset(train_p), batch_size=16, shuffle=True, collate_fn=collate_seqs)
val_loader = DataLoader(OpenFaceBinaryDataset(val_p), batch_size=16, shuffle=False, collate_fn=collate_seqs)
test_loader = DataLoader(OpenFaceBinaryDataset(test_p), batch_size=16, shuffle=False, collate_fn=collate_seqs)

print(f"Train samples: {len(train_p)} | Val samples: {len(val_p)} | Test samples: {len(test_p)}")

# Setup Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_acc = 0.0
best_checkpoint_path = "best_openface_ablation.pth"
epochs = 35

print("\n--- Starting Ablation Training ---")
for epoch in range(1, epochs + 1):
    # Training Phase
    model.train()
    train_loss, train_corr, total_train = 0.0, 0, 0
    for seqs, lens, y in train_loader:
        seqs, lens, y = seqs.to(device), lens.to(device), y.to(device)
        optimizer.zero_grad()
        preds = model(seqs, lens)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * seqs.size(0)
        train_corr += preds.max(1)[1].eq(y).sum().item()
        total_train += y.size(0)
    
    train_acc = (train_corr / total_train) * 100.0
    
    # Validation Phase
    model.eval()
    val_loss, val_corr, total_val = 0.0, 0, 0
    with torch.no_grad():
        for seqs, lens, y in val_loader:
            seqs, lens, y = seqs.to(device), lens.to(device), y.to(device)
            preds = model(seqs, lens)
            val_loss += criterion(preds, y).item() * seqs.size(0)
            val_corr += preds.max(1)[1].eq(y).sum().item()
            total_val += y.size(0)
            
    val_acc = (val_corr / total_val) * 100.0
    
    # Checkpoint on best validation accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_checkpoint_path)
        star = " 🌟 [NEW BEST VAL ACC]"
    else:
        star = ""
        
    print(f"Epoch [{epoch:02d}/{epochs:02d}] Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%{star}")

print(f"\n🏆 Best Validation Accuracy Achieved: {best_val_acc:.2f}%")

## 📌 Step 6: Evaluate on Test Set & Generate Ablation Results Table
This cell computes:
- **Testing Accuracy**
- **Recall**
- **F1-Score**
- **Precision**

and formats the row to directly fit into your project's ablation comparison table!

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt

# Load best weights
model.load_state_dict(torch.load(best_checkpoint_path))
model.eval()

all_preds = []
all_targets = []
all_probs = []

with torch.no_grad():
    for seqs, lens, y in test_loader:
        seqs, lens = seqs.to(device), lens.to(device)
        outputs = model(seqs, lens)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.max(1)[1]
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
all_probs = np.array(all_probs)

# Compute metrics
test_acc = accuracy_score(all_targets, all_preds) * 100.0
precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='weighted', zero_division=0)
precision *= 100.0
recall *= 100.0
f1 *= 100.0

# ------------------------------------------------------------------
# Display Output Formatted Exactly as Your Ablation Table
# ------------------------------------------------------------------
print("=" * 95)
print("             📊 ABLATION STUDY RESULTS TABLE (FOR YOUR REPORT/PAPER) 📊              ")
print("=" * 95)
headers = ["Model", "Best Validation Accuracy", "Testing Accuracy", "Recall", "F1-Score", "Precision"]
fmt = "{:<35} | {:<24} | {:<16} | {:<10} | {:<10} | {:<10}"
print(fmt.format(*headers))
print("-" * 95)
row_name = "OpenFace Alone (Binary Ablation)"
print(fmt.format(row_name, f"{best_val_acc:.2f}%", f"{test_acc:.2f}%", f"{recall:.2f}%", f"{f1:.2f}%", f"{precision:.2f}%"))
print("=" * 95)

print("\nDetailed Classification Report:")
print(classification_report(all_targets, all_preds, target_names=["Engaged", "Disengaged"], digits=4, zero_division=0))

# Confusion Matrix Plot
cm = confusion_matrix(all_targets, all_preds, labels=[0, 1])
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

im = ax[0].imshow(cm, cmap=plt.cm.Blues)
ax[0].set_title("Confusion Matrix (OpenFace Alone)")
ax[0].set_xticks([0, 1])
ax[0].set_yticks([0, 1])
ax[0].set_xticklabels(["Engaged", "Disengaged"])
ax[0].set_yticklabels(["Engaged", "Disengaged"])
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=12)
ax[0].set_xlabel("Predicted")
ax[0].set_ylabel("Ground Truth")

# ROC Curve Plot
if len(np.unique(all_targets)) > 1:
    fpr, tpr, _ = roc_curve(all_targets, all_probs[:, 1])
    roc_auc = auc(fpr, tpr)
    ax[1].plot(fpr, tpr, color='#1f77b4', lw=2.5, label=f'ROC (AUC = {roc_auc:.2f})')
ax[1].plot([0, 1], [0, 1], 'k--', lw=1.5)
ax[1].set_title("ROC Curve")
ax[1].set_xlabel("False Positive Rate")
ax[1].set_ylabel("True Positive Rate")
ax[1].legend(loc="lower right")
ax[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()